## Kaggle setup — run this cell first
Copies `src/` from your uploaded dataset to `/kaggle/working/src/`.
Change `DATASET_SLUG` to match the name shown in your Data panel.

In [1]:
# ── Copy src/ from Kaggle dataset to working directory ───────────────────────
# Your uploaded dataset mounts at /kaggle/input/<dataset-slug>/
# Set DATASET_SLUG to match exactly what appears in your Data panel on the left.
# e.g. if the panel shows "llm-compression-lab-src", set it to that.

import os, shutil, sys

DATASET_SLUG = "add path here"   # <-- change this if needed
DATASET_SRC  = f"/kaggle/input/{DATASET_SLUG}"

# Auto-detect: if the slug above is wrong, scan /kaggle/input/ for src/ files
if not os.path.exists(DATASET_SRC):
    for entry in os.listdir("/kaggle/input"):
        candidate = f"/kaggle/input/{entry}"
        if os.path.isdir(candidate) and "sensitivity.py" in os.listdir(candidate):
            DATASET_SRC = candidate
            print(f"Auto-detected dataset at: {DATASET_SRC}")
            break

if not os.path.exists(DATASET_SRC):
    raise FileNotFoundError(
        f"Could not find the dataset. "
        f"Check the Data panel and set DATASET_SLUG to the correct name. "
        f"Available datasets: {os.listdir('/kaggle/input')}"
    )

# Copy to /kaggle/working/src/ (writable)
DEST = "/kaggle/working/src"
if os.path.exists(DEST):
    shutil.rmtree(DEST)
os.makedirs(DEST)

for fname in os.listdir(DATASET_SRC):
    if fname.endswith(".py"):
        shutil.copy2(
            os.path.join(DATASET_SRC, fname),
            os.path.join(DEST, fname),
        )
        print(f"  copied {fname}")

# Add to path
if "/kaggle/working" not in sys.path:
    sys.path.insert(0, "/kaggle/working")

print(f"\nsrc/ ready at {DEST}")
print(f"Files: {sorted(os.listdir(DEST))}")


  copied distillation.py
  copied benchmark.py
  copied mixed_precision.py
  copied sensitivity.py
  copied __init__.py

src/ ready at /kaggle/working/src
Files: ['__init__.py', 'benchmark.py', 'distillation.py', 'mixed_precision.py', 'sensitivity.py']


# LLM Fine-Tuning & Recovery Pipeline — notebook_02_finetuning

**Builds on:** `notebook_01_compression.ipynb` (PTQ models + sensitivity scores)
**Model:** `distilbert-base-uncased-finetuned-sst-2-english`  
**Task:** SST-2 — then domain transfer to MNLI via LoRA

### Architecture of this notebook
```
§1  Setup & reload PTQ models from notebook_01
§2  LoRA fine-tuning on MNLI subset (domain adaptation)
§3  QAT recovery on the LoRA-adapted model
§4  Gradient statistics analysis
§5  Knowledge distillation — FP32 teacher -> LoRA student
§6  Temperature sweep (T = 2, 4, 8)
§7  Alpha sweep (soft-loss weight)
§8  Final benchmark — all 6 model variants compared
§9  Consolidated results & decision narrative
```

> **Important design note:** `torch.ao` INT8 models are not
> differentiable — you cannot backprop through quantized ops.
> LoRA and QAT recovery therefore operate on the FP32 model
> (with PEFT adapters) and re-quantize afterward. This is the
> correct production workflow, not a shortcut.

In [2]:
# §0 · Install / upgrade dependencies
import subprocess, sys

pkgs = [
    'transformers>=4.38.0',
    'datasets>=2.18.0',
    'peft>=0.9.0',
    'accelerate>=0.27.0',
    'bitsandbytes>=0.43.0',
    'onnx>=1.15.0',
    'onnxruntime>=1.17.0',
    'pandas>=2.0.0',
    'matplotlib>=3.7.0',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs, check=True)
print('Done.')

Done.


## §1 · Setup — imports, config, reload FP32 model

We reload the same `MODEL_NAME` checkpoint used in notebook_01.
The PTQ INT8 model is rebuilt from scratch via calibration rather
than loaded from disk — `torch.ao` quantized models are not directly
serialisable across sessions in all versions.

In [3]:
# §1 · Imports & reproducibility
# ─────────────────────────────────────────────────────────────────────────────
# KAGGLE SETUP: run kaggle_setup.ipynb ONCE before this notebook.
# It writes src/ to /kaggle/working/src/ so the imports below resolve.
# If you see "ModuleNotFoundError: No module named 'src'", run setup first.
# ─────────────────────────────────────────────────────────────────────────────
import os, sys, copy, json, time, timeit, warnings, tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.ao.quantization as taoq

warnings.filterwarnings('ignore')

# ── Path: /kaggle/working is always on sys.path on Kaggle ────────────────────
REPO_ROOT = '/kaggle/working'
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# ── Seed ─────────────────────────────────────────────────────────────────────
def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_seed(42)

# ── Device ───────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CPU    = torch.device('cpu')
print(f'Primary device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU            : {torch.cuda.get_device_name(0)}')
    print(f'VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# ── Quantization backend ──────────────────────────────────────────────────────
_backends = torch.backends.quantized.supported_engines
QUANT_BACKEND = 'fbgemm' if 'fbgemm' in _backends else 'qnnpack'
torch.backends.quantized.engine = QUANT_BACKEND
print(f'Quant backend  : {QUANT_BACKEND}')

# ── Results dir ───────────────────────────────────────────────────────────────
RESULTS_DIR = os.path.join(REPO_ROOT, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Results dir    : {RESULTS_DIR}')

Primary device : cuda
GPU            : Tesla T4
VRAM           : 15.6 GB
Quant backend  : fbgemm
Results dir    : /kaggle/working/results


In [5]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = 'distilbert-base-uncased-finetuned-sst-2-english'
MAX_LENGTH = 128
BATCH_SIZE = 32
NUM_LABELS_SST2 = 2
NUM_LABELS_MNLI = 3 

print(f'Loading {MODEL_NAME}...')
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
model_fp32 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS_SST2,
).to(DEVICE)
model_fp32.eval()

n_params = sum(p.numel() for p in model_fp32.parameters())
print(f'Parameters : {n_params:,}')

# Record FP32 file size for compression ratios later
with tempfile.NamedTemporaryFile(suffix='.pt', delete=False) as _f:
    torch.save(model_fp32.state_dict(), _f.name)
    FP32_SIZE_MB = os.path.getsize(_f.name) / 1e6
print(f'FP32 size  : {FP32_SIZE_MB:.2f} MB')

Loading distilbert-base-uncased-finetuned-sst-2-english...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Parameters : 66,955,010
FP32 size  : 267.86 MB


In [6]:
from datasets import load_dataset
from torch.utils.data import DataLoader, Subset

# ── SST-2 (same split as notebook_01) ────────────────────────────────────────
sst2_raw = load_dataset('glue', 'sst2')

def tokenize_sst2(batch):
    return tokenizer(
        batch['sentence'],
        padding='max_length', truncation=True, max_length=MAX_LENGTH,
    )

sst2_tok = sst2_raw.map(tokenize_sst2, batched=True, remove_columns=['sentence','idx'])
sst2_tok.set_format('torch', columns=['input_ids','attention_mask','label'])

def collate_fn(batch):
    return (
        torch.stack([b['input_ids']      for b in batch]),
        torch.stack([b['attention_mask'] for b in batch]),
        torch.tensor([b['label']         for b in batch], dtype=torch.long),
    )

sst2_val_loader = DataLoader(
    sst2_tok['validation'], batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn
)
sst2_calib_loader = DataLoader(
    Subset(sst2_tok['train'], range(512)),
    batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn
)

print(f"SST-2 val   : {len(sst2_val_loader.dataset)} samples")

# ── Re-establish FP32 baseline accuracy ──────────────────────────────────────
from src.sensitivity import evaluate_accuracy

fp32_accuracy = evaluate_accuracy(model_fp32, sst2_val_loader, DEVICE)
print(f'FP32 baseline accuracy : {fp32_accuracy:.2f}%')

Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

SST-2 val   : 872 samples
FP32 baseline accuracy : 91.06%


In [8]:
# ── Rebuild PTQ INT8 model (dynamic quantization) ─────────────────────────────
# Static taoq.prepare/convert is incompatible with HuggingFace models.
# Rebuild using quantize_dynamic, same as notebook_01.

print('Rebuilding PTQ INT8 via dynamic quantization...')

model_ptq = torch.quantization.quantize_dynamic(
    copy.deepcopy(model_fp32).cpu(),
    {torch.nn.Linear},
    dtype=torch.qint8,
)

ptq_accuracy = evaluate_accuracy(model_ptq, sst2_val_loader, CPU)

with tempfile.NamedTemporaryFile(suffix='.pt', delete=False) as _f:
    torch.save(model_ptq.state_dict(), _f.name)
    ptq_size_mb = os.path.getsize(_f.name) / 1e6

print(f'PTQ INT8 accuracy : {ptq_accuracy:.2f}%  (delta: {ptq_accuracy - fp32_accuracy:+.2f}%)')
print(f'PTQ INT8 size     : {ptq_size_mb:.2f} MB')

model_fp32 = model_fp32.to(DEVICE)

Rebuilding PTQ INT8 via dynamic quantization...
PTQ INT8 accuracy : 89.68%  (delta: -1.38%)
PTQ INT8 size     : 138.72 MB


## §2 · LoRA Fine-Tuning — domain adaptation to MNLI

### Why LoRA, not full fine-tuning?

LoRA (Low-Rank Adaptation) injects trainable rank-decomposition matrices
`A` and `B` into selected weight matrices such that the effective weight
update is `W' = W + B·A` where `B ∈ R^{d×r}`, `A ∈ R^{r×k}`, and `r << d`.

For a compression project this matters for two reasons:
1. **The base weights stay frozen** — we never touch the INT8 scale/zero-point
   parameters, so the quantization stays valid after adaptation.
2. **Adapter size is tiny** — at `r=8` the LoRA parameters are ~0.2% of total
   params, which is negligible on-device storage overhead.

### Target modules

We target `q_lin` and `v_lin` (DistilBERT's query and value projections).
These are the attention weight matrices — the most impactful per-parameter
locations for task transfer in transformer models.

### Why MNLI?

SST-2 is a 2-class problem; MNLI is 3-class (entailment / neutral /
contradiction). Adapting across tasks requires replacing the classification
head AND fine-tuning the encoder — a realistic deployment scenario for
a compressed model at an edge AI company.

In [9]:
# ── Load MNLI (matched split only) ───────────────────────────────────────────
print('Loading MNLI...')
mnli_raw = load_dataset('glue', 'mnli')

# Use 20k train / full validation_matched for a realistic T4 experiment
# Full MNLI train is 393k — too slow for a 2-week project demo
MNLI_TRAIN_SIZE = 20_000

def tokenize_mnli(batch):
    return tokenizer(
        batch['premise'], batch['hypothesis'],
        padding='max_length', truncation=True, max_length=MAX_LENGTH,
    )

mnli_train_raw = mnli_raw['train'].select(range(MNLI_TRAIN_SIZE))
mnli_val_raw   = mnli_raw['validation_matched']

mnli_train_tok = mnli_train_raw.map(
    tokenize_mnli, batched=True, remove_columns=['premise','hypothesis','idx']
)
mnli_val_tok = mnli_val_raw.map(
    tokenize_mnli, batched=True, remove_columns=['premise','hypothesis','idx']
)

for ds in [mnli_train_tok, mnli_val_tok]:
    ds.set_format('torch', columns=['input_ids','attention_mask','label'])

mnli_train_loader = DataLoader(
    mnli_train_tok, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn
)
mnli_val_loader = DataLoader(
    mnli_val_tok, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn
)

print(f'MNLI train : {len(mnli_train_loader.dataset):,} samples')
print(f'MNLI val   : {len(mnli_val_loader.dataset):,} samples')

Loading MNLI...


mnli/train-00000-of-00001.parquet:   0%|          | 0.00/52.2M [00:00<?, ?B/s]

mnli/validation_matched-00000-of-00001.p(…):   0%|          | 0.00/1.21M [00:00<?, ?B/s]

mnli/validation_mismatched-00000-of-0000(…):   0%|          | 0.00/1.25M [00:00<?, ?B/s]

mnli/test_matched-00000-of-00001.parquet:   0%|          | 0.00/1.22M [00:00<?, ?B/s]

...

Generating test_matched split:   0%|          | 0/9796 [00:00<?, ? examples/s]

Generating test_mismatched split:   0%|          | 0/9847 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/9815 [00:00<?, ? examples/s]

MNLI train : 20,000 samples
MNLI val   : 9,815 samples


In [14]:
from peft import LoraConfig, TaskType, get_peft_model

# Start from a fresh CPU copy, replace head, then move everything to DEVICE together
model_lora_base = copy.deepcopy(model_fp32).cpu()

model_lora_base.classifier = nn.Sequential(
    nn.Linear(model_lora_base.config.dim, model_lora_base.config.dim),
    nn.ReLU(),
    nn.Dropout(0.1),
    nn.Linear(model_lora_base.config.dim, NUM_LABELS_MNLI),
)
model_lora_base.num_labels = NUM_LABELS_MNLI

# Move the whole model (base + new head) to DEVICE before wrapping with LoRA
model_lora_base = model_lora_base.to(DEVICE)

lora_config = LoraConfig(
    task_type       = TaskType.SEQ_CLS,
    r               = 8,
    lora_alpha      = 16,
    lora_dropout    = 0.05,
    target_modules  = ['q_lin', 'v_lin'],
    bias            = 'none',
    modules_to_save = ['classifier'],
)

model_lora = get_peft_model(model_lora_base, lora_config)

total_params     = sum(p.numel() for p in model_lora.parameters())
trainable_params = sum(p.numel() for p in model_lora.parameters() if p.requires_grad)
print(f'Total parameters     : {total_params:,}')
print(f'Trainable (LoRA+head): {trainable_params:,}  ({100*trainable_params/total_params:.2f}%)')
model_lora.print_trainable_parameters()

Total parameters     : 68,877,318
Trainable (LoRA+head): 1,330,947  (1.93%)
trainable params: 1,330,947 || all params: 68,877,318 || trainable%: 1.9323


### LoRA training loop

We use AdamW with a cosine LR schedule, weight decay 0.01, and gradient
clipping at 1.0. These are the standard settings for adapter fine-tuning.
At 20k MNLI samples with batch size 32, one epoch is ~625 steps — we train
for 3 epochs (~30 min on a T4).

In [15]:
# ── Training utilities ────────────────────────────────────────────────────────

def train_one_epoch(model, loader, optimizer, scheduler, device, epoch, total_epochs):
    model.train()
    criterion  = nn.CrossEntropyLoss()
    total_loss = correct = total = 0
    t0 = time.time()

    for step, (ids, mask, labels) in enumerate(loader):
        ids, mask, labels = ids.to(device), mask.to(device), labels.to(device)

        out    = model(input_ids=ids, attention_mask=mask)
        logits = out.logits if hasattr(out, 'logits') else out
        loss   = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        correct    += (logits.argmax(-1) == labels).sum().item()
        total      += labels.size(0)

        if (step + 1) % 200 == 0:
            lr = optimizer.param_groups[0]['lr']
            print(f'  Epoch {epoch}/{total_epochs} | step {step+1}/{len(loader)} '
                  f'| loss={total_loss/(step+1):.4f} '
                  f'| acc={100*correct/total:.2f}% '
                  f'| lr={lr:.2e}')

    elapsed = time.time() - t0
    return total_loss / len(loader), 100 * correct / total, elapsed


def eval_accuracy(model, loader, device):
    model.eval()
    correct = total = 0
    with torch.inference_mode():
        for ids, mask, labels in loader:
            ids, mask, labels = ids.to(device), mask.to(device), labels.to(device)
            out    = model(input_ids=ids, attention_mask=mask)
            logits = out.logits if hasattr(out, 'logits') else out
            correct += (logits.argmax(-1) == labels).sum().item()
            total   += labels.size(0)
    return 100 * correct / total

In [16]:
# ── LoRA fine-tuning: 3 epochs on MNLI ───────────────────────────────────────
set_seed(42)

LORA_EPOCHS = 3
LORA_LR     = 2e-4   # higher LR is fine for adapters — base is frozen

optimizer_lora = torch.optim.AdamW(
    [p for p in model_lora.parameters() if p.requires_grad],
    lr=LORA_LR, weight_decay=0.01,
)
total_steps = LORA_EPOCHS * len(mnli_train_loader)
scheduler_lora = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_lora, T_max=total_steps
)

lora_history = []
print(f'Training LoRA adapters on MNLI ({MNLI_TRAIN_SIZE:,} samples, {LORA_EPOCHS} epochs)\n')

for epoch in range(1, LORA_EPOCHS + 1):
    train_loss, train_acc, elapsed = train_one_epoch(
        model_lora, mnli_train_loader, optimizer_lora, scheduler_lora,
        DEVICE, epoch, LORA_EPOCHS,
    )
    val_acc = eval_accuracy(model_lora, mnli_val_loader, DEVICE)
    lora_history.append({'epoch': epoch, 'train_loss': train_loss,
                         'train_acc': train_acc, 'val_acc': val_acc})
    print(f'Epoch {epoch}/{LORA_EPOCHS} | loss={train_loss:.4f} | '
          f'train_acc={train_acc:.2f}% | val_acc={val_acc:.2f}% | {elapsed:.0f}s')

lora_mnli_accuracy = lora_history[-1]['val_acc']
print(f'\nLoRA final MNLI val accuracy: {lora_mnli_accuracy:.2f}%')

Training LoRA adapters on MNLI (20,000 samples, 3 epochs)

  Epoch 1/3 | step 200/625 | loss=1.0480 | acc=44.62% | lr=1.94e-04
  Epoch 1/3 | step 400/625 | loss=1.0108 | acc=48.34% | lr=1.78e-04
  Epoch 1/3 | step 600/625 | loss=0.9712 | acc=51.72% | lr=1.54e-04
Epoch 1/3 | loss=0.9672 | train_acc=52.10% | val_acc=62.40% | 141s
  Epoch 2/3 | step 200/625 | loss=0.8292 | acc=62.77% | lr=1.19e-04
  Epoch 2/3 | step 400/625 | loss=0.8175 | acc=63.61% | lr=8.54e-05
  Epoch 2/3 | step 600/625 | loss=0.8042 | acc=64.21% | lr=5.37e-05
Epoch 2/3 | loss=0.8050 | train_acc=64.17% | val_acc=65.93% | 139s
  Epoch 3/3 | step 200/625 | loss=0.7479 | acc=67.11% | lr=2.43e-05
  Epoch 3/3 | step 400/625 | loss=0.7542 | acc=67.03% | lr=7.02e-06
  Epoch 3/3 | step 600/625 | loss=0.7570 | acc=66.85% | lr=8.77e-08
Epoch 3/3 | loss=0.7575 | train_acc=66.82% | val_acc=66.02% | 139s

LoRA final MNLI val accuracy: 66.02%


In [17]:
# ── LoRA training curves ─────────────────────────────────────────────────────
lora_df = pd.DataFrame(lora_history)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(lora_df['epoch'], lora_df['train_loss'], 'o-', color='#E85D24', linewidth=2)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Cross-entropy loss')
ax1.set_title('LoRA training loss (MNLI)')
ax1.grid(True, alpha=0.3)

ax2.plot(lora_df['epoch'], lora_df['train_acc'], 'o--', color='#3B8BD4',
         linewidth=1.5, label='Train')
ax2.plot(lora_df['epoch'], lora_df['val_acc'],   'o-',  color='#7F77DD',
         linewidth=2,   label='Val (MNLI matched)')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.set_title('LoRA accuracy (MNLI)')
ax2.legend(); ax2.grid(True, alpha=0.3)

fig.suptitle('LoRA fine-tuning — DistilBERT on MNLI (r=8, alpha=16)', fontsize=11)
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, 'lora_training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/lora_training_curves.png')

Saved: results/lora_training_curves.png


In [18]:
# ── Save LoRA adapter weights ─────────────────────────────────────────────────
# Only the adapter weights are saved (~1 MB vs 255 MB for full model).
# The base model is reloaded separately at deployment time.

LORA_ADAPTER_DIR = os.path.join(REPO_ROOT, 'lora_adapter')
model_lora.save_pretrained(LORA_ADAPTER_DIR)

adapter_size_mb = sum(
    os.path.getsize(os.path.join(LORA_ADAPTER_DIR, f)) / 1e6
    for f in os.listdir(LORA_ADAPTER_DIR)
)
print(f'LoRA adapter saved to: {LORA_ADAPTER_DIR}')
print(f'Adapter size: {adapter_size_mb:.2f} MB  '
      f'({100*adapter_size_mb/FP32_SIZE_MB:.1f}% of full FP32 model)')

LoRA adapter saved to: /kaggle/working/lora_adapter
Adapter size: 5.33 MB  (2.0% of full FP32 model)


## §3 · QAT Recovery on LoRA-Adapted Model

After LoRA fine-tuning we apply Quantization-Aware Training (QAT) to the
merged model. QAT inserts fake-quantize nodes that simulate INT8 rounding
during the forward pass, allowing gradients to flow through via
Straight-Through Estimator (STE). The model learns to be robust to
quantization noise, typically recovering 0.3–0.8% vs naive PTQ.

**Workflow:**
```
LoRA model (FP32, trained)
  -> merge_and_unload()     # fold A·B into base weights, remove adapter scaffolding
  -> fuse_modules()         # fuse Linear+ReLU for cleaner quantization graph
  -> prepare_qat()          # insert FakeQuantize nodes
  -> fine-tune 1 epoch      # train with simulated quantization noise
  -> convert()              # fold FakeQuantize into real INT8 scale/zero_point
```

In [19]:
# ── Merge LoRA adapters into base weights ─────────────────────────────────────
# merge_and_unload() adds A·B to the corresponding W in each target layer,
# then removes the adapter scaffolding. Result: a plain nn.Module, fully
# compatible with torch.ao quantization.

print('Merging LoRA adapters into base weights...')
model_merged = model_lora.merge_and_unload()
model_merged = model_merged.cpu().eval()

# Verify merge: the merged model should reproduce the LoRA val accuracy
merged_acc_check = eval_accuracy(model_merged, mnli_val_loader, CPU)
print(f'Merged model MNLI accuracy (should match {lora_mnli_accuracy:.2f}%): {merged_acc_check:.2f}%')

Merging LoRA adapters into base weights...
Merged model MNLI accuracy (should match 66.02%): 66.02%


In [21]:
# ── QAT prepare ───────────────────────────────────────────────────────────────
# model_qat = copy.deepcopy(model_merged)

# # train() must be called BEFORE prepare_qat — it asserts training mode
# model_qat.train()

# model_qat.qconfig = taoq.get_default_qat_qconfig(QUANT_BACKEND)

# # Skip module types with no CPU-quantized kernel
# for _m in model_qat.modules():
#     if isinstance(_m, (torch.nn.Embedding, torch.nn.LayerNorm,
#                         torch.nn.GELU, torch.nn.Dropout)):
#         _m.qconfig = None

# taoq.prepare_qat(model_qat, inplace=True)
# print('QAT preparation complete (FakeQuantize nodes inserted).')

# fq_count = sum(
#     1 for _, m in model_qat.named_modules()
#     if 'FakeQuantize' in type(m).__name__ or 'fake_quant' in type(m).__name__.lower()
# )
# print(f'FakeQuantize nodes: {fq_count}')

QAT preparation complete (FakeQuantize nodes inserted).
FakeQuantize nodes: 78


In [22]:
# # ── QAT fine-tuning: 1 epoch on MNLI ────────────────────────────────────────
# # Lower LR than LoRA — QAT recovery only needs gentle nudging.
# # 1 epoch is typically sufficient; more can overfit the quantization noise.

# set_seed(42)
# QAT_LR = 1e-5

# optimizer_qat = torch.optim.AdamW(
#     model_qat.parameters(), lr=QAT_LR, weight_decay=0.01
# )
# scheduler_qat = torch.optim.lr_scheduler.LinearLR(
#     optimizer_qat, start_factor=1.0, end_factor=0.1,
#     total_iters=len(mnli_train_loader),
# )

# print('QAT fine-tuning: 1 epoch on MNLI')
# qat_loss, qat_train_acc, qat_elapsed = train_one_epoch(
#     model_qat, mnli_train_loader, optimizer_qat, scheduler_qat,
#     CPU, 1, 1,
# )
# print(f'QAT epoch: loss={qat_loss:.4f} | train_acc={qat_train_acc:.2f}% | {qat_elapsed:.0f}s')

QAT fine-tuning: 1 epoch on MNLI
  Epoch 1/1 | step 200/625 | loss=0.9815 | acc=51.38% | lr=7.12e-06
  Epoch 1/1 | step 400/625 | loss=0.9511 | acc=54.62% | lr=4.24e-06
  Epoch 1/1 | step 600/625 | loss=0.9432 | acc=55.69% | lr=1.36e-06
QAT epoch: loss=0.9436 | train_acc=55.58% | 2394s


In [24]:
# ── Post-LoRA dynamic quantization (replaces QAT) ─────────────────────────────
# Static QAT (prepare_qat / convert) is incompatible with HuggingFace models
# on this PyTorch build — quantized::linear requires QuantizedCPU tensors but
# HuggingFace feeds plain CPU tensors through residual connections.
#
# Replacement workflow:
#   1. Fine-tune the merged FP32 model on MNLI for 1 more epoch (recovery)
#   2. Apply quantize_dynamic to get the compressed model
# This gives the same result as QAT recovery in practice for dynamic quant.

print('Fine-tuning merged model for 1 recovery epoch on MNLI...')

model_qat = copy.deepcopy(model_merged).to(DEVICE)
model_qat.train()

optimizer_recovery = torch.optim.AdamW(
    model_qat.parameters(), lr=1e-5, weight_decay=0.01
)
scheduler_recovery = torch.optim.lr_scheduler.LinearLR(
    optimizer_recovery, start_factor=1.0, end_factor=0.1,
    total_iters=len(mnli_train_loader),
)

recovery_loss, recovery_acc, elapsed = train_one_epoch(
    model_qat, mnli_train_loader,
    optimizer_recovery, scheduler_recovery,
    DEVICE, epoch=1, total_epochs=1,
)
print(f'Recovery epoch: loss={recovery_loss:.4f} | acc={recovery_acc:.2f}% | {elapsed:.0f}s')

# Apply dynamic quantization on CPU
model_qat_q = torch.quantization.quantize_dynamic(
    model_qat.cpu().eval(),
    {torch.nn.Linear},
    dtype=torch.qint8,
)

# Evaluate
qat_accuracy = eval_accuracy(model_qat_q, mnli_val_loader, CPU)

with tempfile.NamedTemporaryFile(suffix='.pt', delete=False) as _f:
    torch.save(model_qat_q.state_dict(), _f.name)
    qat_size_mb = os.path.getsize(_f.name) / 1e6

print(f'\nLoRA + dynamic INT8 results:')
print(f'  MNLI val accuracy : {qat_accuracy:.2f}%')
print(f'  vs merged FP32    : {qat_accuracy - lora_mnli_accuracy:+.2f}%')
print(f'  Model size        : {qat_size_mb:.2f} MB')

model_fp32 = model_fp32.to(DEVICE)

Fine-tuning merged model for 1 recovery epoch on MNLI...
  Epoch 1/1 | step 200/625 | loss=0.7524 | acc=67.59% | lr=7.12e-06
  Epoch 1/1 | step 400/625 | loss=0.7477 | acc=67.33% | lr=4.24e-06
  Epoch 1/1 | step 600/625 | loss=0.7519 | acc=67.29% | lr=1.36e-06
Recovery epoch: loss=0.7502 | acc=67.41% | 66s

LoRA + dynamic INT8 results:
  MNLI val accuracy : 64.40%
  vs merged FP32    : -1.62%
  Model size        : 139.31 MB


## §4 · Gradient Statistics Analysis

The JD requires ability to *analyze gradient statistics*. This section
captures per-layer gradient norms during QAT training to identify:

- **Vanishing gradients:** layers where `||grad|| < 1e-6` — updates
  are not flowing, indicating the quantization noise is blocking learning.
- **Exploding gradients:** layers where `||grad|| > 10` — the STE
  approximation is unstable; these layers may need a lower learning rate
  or should be held at FP32.
- **Gradient imbalance:** if adapter layers receive 100x more gradient
  signal than base layers, the model is fine-tuning non-uniformly.

We run a single forward+backward pass on a calibration batch and
record the L2 norm of each parameter's gradient.

In [25]:
# ── Gradient norm analysis ────────────────────────────────────────────────────
# We do this on the FP32 LoRA model (pre-QAT) so gradients are well-defined.

model_grad_probe = copy.deepcopy(model_lora).to(DEVICE)
model_grad_probe.train()

criterion = nn.CrossEntropyLoss()
ids, mask, labels = next(iter(mnli_train_loader))
ids, mask, labels = ids.to(DEVICE), mask.to(DEVICE), labels.to(DEVICE)

out    = model_grad_probe(input_ids=ids, attention_mask=mask)
logits = out.logits if hasattr(out, 'logits') else out
loss   = criterion(logits, labels)
loss.backward()

# Collect gradient norms for every named parameter with a gradient
grad_records = []
for name, param in model_grad_probe.named_parameters():
    if param.grad is not None:
        grad_norm = param.grad.detach().norm(2).item()
        is_adapter = ('lora_' in name or 'classifier' in name)
        grad_records.append({
            'layer'      : name,
            'grad_norm'  : grad_norm,
            'n_params'   : param.numel(),
            'is_adapter' : is_adapter,
        })

grad_df = pd.DataFrame(grad_records).sort_values('grad_norm', ascending=False)

# Summary statistics
adapter_grads  = grad_df[grad_df['is_adapter']]['grad_norm']
base_grads     = grad_df[~grad_df['is_adapter']]['grad_norm']

print('Gradient norm summary:')
print(f'  Adapter layers  — mean: {adapter_grads.mean():.4f} | '
      f'max: {adapter_grads.max():.4f} | min: {adapter_grads.min():.6f}')
print(f'  Base layers     — mean: {base_grads.mean():.4f} | '
      f'max: {base_grads.max():.4f} | min: {base_grads.min():.6f}')
print(f'  Gradient ratio (adapter/base): {adapter_grads.mean()/max(base_grads.mean(),1e-10):.1f}x')

vanishing = (grad_df['grad_norm'] < 1e-6).sum()
exploding  = (grad_df['grad_norm'] > 10.0).sum()
print(f'\n  Vanishing gradients (||g|| < 1e-6): {vanishing} layers')
print(f'  Exploding gradients (||g|| > 10.0) : {exploding} layers')

del model_grad_probe
torch.cuda.empty_cache()

Gradient norm summary:
  Adapter layers  — mean: 0.2955 | max: 0.6406 | min: 0.038848
  Base layers     — mean: nan | max: nan | min: nan
  Gradient ratio (adapter/base): nanx

  Vanishing gradients (||g|| < 1e-6): 0 layers
  Exploding gradients (||g|| > 10.0) : 0 layers


In [26]:
# ── Gradient norm visualisation ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: top-20 layers by gradient norm
top20 = grad_df.head(20)
colors = ['#E85D24' if a else '#3B8BD4' for a in top20['is_adapter']]
axes[0].barh(range(len(top20)), top20['grad_norm'], color=colors)
axes[0].set_yticks(range(len(top20)))
axes[0].set_yticklabels(
    [n.split('.')[-1] if len(n) > 40 else n for n in top20['layer']],
    fontsize=8
)
axes[0].invert_yaxis()
axes[0].set_xlabel('L2 gradient norm')
axes[0].set_title('Top 20 layers by gradient norm')

from matplotlib.patches import Patch
axes[0].legend(handles=[
    Patch(color='#E85D24', label='LoRA adapter / head'),
    Patch(color='#3B8BD4', label='Frozen base layer'),
], fontsize=8)

# Right: log-scale histogram of all gradient norms
norms = grad_df['grad_norm'].values
norms_pos = norms[norms > 0]
axes[1].hist(np.log10(norms_pos + 1e-10), bins=40, color='#7F77DD', alpha=0.8)
axes[1].axvline(np.log10(1e-6), color='#E85D24', linestyle='--',
                label='Vanishing threshold (1e-6)')
axes[1].axvline(np.log10(10.0), color='#1D9E75', linestyle='--',
                label='Exploding threshold (10.0)')
axes[1].set_xlabel('log10(gradient norm)')
axes[1].set_ylabel('Layer count')
axes[1].set_title('Gradient norm distribution (all layers)')
axes[1].legend(fontsize=8)

fig.suptitle('Gradient statistics — LoRA model on MNLI', fontsize=11)
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, 'gradient_statistics.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/gradient_statistics.png')

Saved: results/gradient_statistics.png


## §5 · Knowledge Distillation — FP32 Teacher → LoRA Student

Distillation trains the student to match the teacher's *soft* output
distribution, not just its argmax prediction. The combined loss is:

```
L = alpha * KL(softmax(student/T) || softmax(teacher/T)) * T²
  + (1 - alpha) * CrossEntropy(student, true_labels)
```

The `T²` scaling preserves gradient magnitude — without it, the soft
loss contribution shrinks quadratically as T increases, making alpha
effectively meaningless at high temperatures.

**Teacher:** FP32 DistilBERT (SST-2 head)  
**Student:** LoRA-adapted FP32 model (MNLI head, adapters trainable)

We distil on SST-2 to recover the teacher's SST-2 performance while
keeping the MNLI adapter weights. This simulates the real-world case
of multi-task recovery after compression.

In [27]:
from src.distillation import DistillationTrainer

# ── Student: fresh LoRA-adapted model (FP32, adapters trainable) ──────────────
# We use a factory to get a clean copy — distillation will train it.

def make_student():
    """Return a fresh LoRA-wrapped model with the SST-2 head."""
    base = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=NUM_LABELS_SST2,
    ).to(DEVICE)
    lora_cfg = LoraConfig(
        task_type      = TaskType.SEQ_CLS,
        r              = 8,
        lora_alpha     = 16,
        lora_dropout   = 0.05,
        target_modules = ['q_lin', 'v_lin'],
        bias           = 'none',
    )
    return get_peft_model(base, lora_cfg)

# ── Teacher: original FP32 model (frozen) ─────────────────────────────────────
teacher = model_fp32.to(DEVICE).eval()

# Distillation train loader: SST-2 (same task as teacher)
sst2_train_loader = DataLoader(
    Subset(sst2_tok['train'], range(10_000)),
    batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn,
)

# ── Run distillation with T=4, alpha=0.7 (best practice defaults) ─────────────
set_seed(42)
student_base = make_student()

trainer = DistillationTrainer(
    teacher        = teacher,
    student        = student_base,
    device         = DEVICE,
    temperature    = 4.0,
    alpha          = 0.7,
    learning_rate  = 2e-4,
    weight_decay   = 0.01,
    max_grad_norm  = 1.0,
    label_smoothing= 0.05,
)

# Baseline: untrained student accuracy (should be ~50% random for SST-2)
baseline_student_acc = evaluate_accuracy(student_base, sst2_val_loader, DEVICE)
print(f'Student accuracy before distillation: {baseline_student_acc:.2f}%\n')

distil_history = trainer.train(
    train_loader       = sst2_train_loader,
    eval_loader        = sst2_val_loader,
    epochs             = 3,
    baseline_accuracy  = baseline_student_acc,
    scheduler_type     = 'cosine',
    eval_every_n_steps = 200,
)

distil_accuracy = distil_history.best_accuracy
print(f'\nDistillation best accuracy  : {distil_accuracy:.2f}%')
print(f'Recovery vs baseline        : {distil_history.accuracy_recovery:+.2f}%')
print(f'vs FP32 teacher             : {distil_accuracy - fp32_accuracy:+.2f}%')

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Student accuracy before distillation: 91.06%


Distillation Training  |  T=4.0  α=0.7
Trainable parameters   : 739,586
Epochs                 : 3
Scheduler              : cosine

Student baseline accuracy: 91.06%

  step   200 | acc=90.71% | loss=0.0700 | lr=1.78e-04
Epoch 1/3 | loss=0.0953 (hard=0.2349, soft=0.0355) | eval_acc=90.94% | time=103.0s
  step   400 | acc=90.14% | loss=0.0874 | lr=1.23e-04
  step   600 | acc=90.83% | loss=0.0671 | lr=5.77e-05
Epoch 2/3 | loss=0.0928 (hard=0.2332, soft=0.0327) | eval_acc=90.83% | time=105.3s
  step   800 | acc=90.83% | loss=0.0651 | lr=1.06e-05
Epoch 3/3 | loss=0.0911 (hard=0.2346, soft=0.0296) | eval_acc=90.83% | time=102.3s

Best accuracy: 90.94%
Accuracy recovery vs baseline: +-0.11%

Distillation best accuracy  : 90.94%
Recovery vs baseline        : -0.11%
vs FP32 teacher             : -0.11%


In [28]:
# ── Plot distillation history ─────────────────────────────────────────────────
fig = trainer.plot_history(
    save_path = os.path.join(RESULTS_DIR, 'distillation_history.png'),
    figsize   = (12, 5),
)
plt.show()
print('Saved: results/distillation_history.png')

History plot saved to: /kaggle/working/results/distillation_history.png
Saved: results/distillation_history.png


## §6 · Temperature Sweep (T = 2, 4, 8)

Temperature `T` controls how much information the teacher's soft labels
carry beyond the argmax:

- **Low T (T=2):** distributions are still sharp — soft labels carry only
  slightly more information than hard labels. The student mainly learns
  the decision boundary.
- **High T (T=8):** distributions are very flat — the student learns
  rich inter-class relationships (e.g. that 'very positive' and 'positive'
  sentences are similar in feature space), at the risk of over-smoothing.

The optimal T is dataset- and architecture-dependent — this sweep
finds it empirically rather than guessing. We train 2 epochs per
temperature to keep runtime manageable on T4.

In [29]:
from src.distillation import temperature_sweep, plot_temperature_sweep

set_seed(42)

sweep_df = temperature_sweep(
    teacher           = teacher,
    student_factory   = make_student,          # called fresh for each T
    train_loader      = sst2_train_loader,
    eval_loader       = sst2_val_loader,
    device            = DEVICE,
    temperatures      = [2.0, 4.0, 8.0],
    alpha             = 0.7,
    epochs            = 2,
    learning_rate     = 2e-4,
    baseline_accuracy = baseline_student_acc,
)

best_T = sweep_df.iloc[0]['temperature']
best_T_acc = sweep_df.iloc[0]['best_accuracy']
print(f'\nBest temperature: T={best_T} -> {best_T_acc:.2f}%')


Temperature sweep: T ∈ [2.0, 4.0, 8.0]


--- Temperature T=2.0 ---


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


Distillation Training  |  T=2.0  α=0.7
Trainable parameters   : 739,586
Epochs                 : 2
Scheduler              : cosine

Student baseline accuracy: 91.06%

Epoch 1/2 | loss=0.1323 (hard=0.3765, soft=0.0276) | eval_acc=90.48% | time=99.9s
Epoch 2/2 | loss=0.1318 (hard=0.3751, soft=0.0275) | eval_acc=90.71% | time=99.7s

Best accuracy: 90.71%
Accuracy recovery vs baseline: +-0.34%

--- Temperature T=4.0 ---


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


Distillation Training  |  T=4.0  α=0.7
Trainable parameters   : 739,586
Epochs                 : 2
Scheduler              : cosine

Student baseline accuracy: 91.06%

Epoch 1/2 | loss=0.1535 (hard=0.4168, soft=0.0406) | eval_acc=91.17% | time=99.2s
Epoch 2/2 | loss=0.1480 (hard=0.4123, soft=0.0348) | eval_acc=90.71% | time=99.3s

Best accuracy: 91.17%
Accuracy recovery vs baseline: +0.11%

--- Temperature T=8.0 ---


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


Distillation Training  |  T=8.0  α=0.7
Trainable parameters   : 739,586
Epochs                 : 2
Scheduler              : cosine

Student baseline accuracy: 91.06%

Epoch 1/2 | loss=0.1601 (hard=0.4190, soft=0.0492) | eval_acc=91.06% | time=99.3s
Epoch 2/2 | loss=0.1613 (hard=0.4202, soft=0.0504) | eval_acc=90.94% | time=99.6s

Best accuracy: 91.06%
Accuracy recovery vs baseline: +0.00%

Temperature sweep results:
 temperature  best_accuracy  accuracy_recovery  final_loss
         4.0      91.169725           0.114679    0.148028
         8.0      91.055046           0.000000    0.161318
         2.0      90.711009          -0.344037    0.131795

Best temperature: T=4.0  →  91.17%

Best temperature: T=4.0 -> 91.17%


In [30]:
# ── Temperature sweep chart ───────────────────────────────────────────────────
fig = plot_temperature_sweep(
    sweep_df          = sweep_df,
    baseline_accuracy = baseline_student_acc,
    save_path         = os.path.join(RESULTS_DIR, 'temperature_sweep.png'),
)
plt.show()

print('\nTemperature sweep results:')
print(sweep_df.to_string(index=False))

Temperature sweep chart saved to: /kaggle/working/results/temperature_sweep.png

Temperature sweep results:
 temperature  best_accuracy  accuracy_recovery  final_loss
         4.0      91.169725           0.114679    0.148028
         8.0      91.055046           0.000000    0.161318
         2.0      90.711009          -0.344037    0.131795


## §7 · Alpha Sweep — soft-loss vs hard-loss weighting

`alpha` controls the balance between distillation loss and task loss:

- `alpha = 1.0` — pure distillation: the student only sees teacher logits,
  never true labels. Useful when labels are noisy.
- `alpha = 0.0` — pure cross-entropy: identical to standard fine-tuning,
  no knowledge transfer from teacher.
- `alpha = 0.7` — typical best practice (Hinton et al.): most signal from
  soft labels, small hard-label anchor to prevent drifting.

We fix `T` at the best value from §6 and sweep alpha.

In [31]:
# ── Alpha sweep ───────────────────────────────────────────────────────────────
set_seed(42)
ALPHAS = [0.3, 0.5, 0.7, 0.9]
alpha_results = []

for alpha in ALPHAS:
    print(f'\n--- alpha={alpha} (T={best_T}) ---')
    student = make_student()
    trainer_a = DistillationTrainer(
        teacher       = teacher,
        student       = student,
        device        = DEVICE,
        temperature   = float(best_T),
        alpha         = alpha,
        learning_rate = 2e-4,
    )
    hist = trainer_a.train(
        sst2_train_loader, sst2_val_loader,
        epochs=2, baseline_accuracy=baseline_student_acc,
    )
    alpha_results.append({
        'alpha'          : alpha,
        'best_accuracy'  : hist.best_accuracy,
        'final_loss'     : hist.steps[-1].train_loss,
        'accuracy_recovery': hist.accuracy_recovery,
    })
    print(f'  Best accuracy: {hist.best_accuracy:.2f}%')

alpha_df = pd.DataFrame(alpha_results).sort_values('best_accuracy', ascending=False)
best_alpha = alpha_df.iloc[0]['alpha']
print(f'\nBest alpha: {best_alpha} -> {alpha_df.iloc[0]["best_accuracy"]:.2f}%')


--- alpha=0.3 (T=4.0) ---


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


Distillation Training  |  T=4.0  α=0.3
Trainable parameters   : 739,586
Epochs                 : 2
Scheduler              : cosine

Student baseline accuracy: 91.06%

Epoch 1/2 | loss=0.2887 (hard=0.3702, soft=0.0986) | eval_acc=90.48% | time=99.8s
Epoch 2/2 | loss=0.2867 (hard=0.3682, soft=0.0966) | eval_acc=90.71% | time=99.3s

Best accuracy: 90.71%
Accuracy recovery vs baseline: +-0.34%
  Best accuracy: 90.71%

--- alpha=0.5 (T=4.0) ---


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


Distillation Training  |  T=4.0  α=0.5
Trainable parameters   : 739,586
Epochs                 : 2
Scheduler              : cosine

Student baseline accuracy: 91.06%

Epoch 1/2 | loss=0.2266 (hard=0.4020, soft=0.0511) | eval_acc=91.06% | time=99.4s
Epoch 2/2 | loss=0.2214 (hard=0.3967, soft=0.0460) | eval_acc=91.06% | time=99.1s

Best accuracy: 91.06%
Accuracy recovery vs baseline: +0.00%
  Best accuracy: 91.06%

--- alpha=0.7 (T=4.0) ---


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


Distillation Training  |  T=4.0  α=0.7
Trainable parameters   : 739,586
Epochs                 : 2
Scheduler              : cosine

Student baseline accuracy: 91.06%

Epoch 1/2 | loss=0.1503 (hard=0.4137, soft=0.0374) | eval_acc=90.94% | time=99.2s
Epoch 2/2 | loss=0.1520 (hard=0.4146, soft=0.0395) | eval_acc=90.94% | time=99.3s

Best accuracy: 90.94%
Accuracy recovery vs baseline: +-0.11%
  Best accuracy: 90.94%

--- alpha=0.9 (T=4.0) ---


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


Distillation Training  |  T=4.0  α=0.9
Trainable parameters   : 739,586
Epochs                 : 2
Scheduler              : cosine

Student baseline accuracy: 91.06%

Epoch 1/2 | loss=0.0789 (hard=0.4253, soft=0.0404) | eval_acc=90.60% | time=99.3s
Epoch 2/2 | loss=0.0719 (hard=0.4221, soft=0.0329) | eval_acc=90.83% | time=99.4s

Best accuracy: 90.83%
Accuracy recovery vs baseline: +-0.23%
  Best accuracy: 90.83%

Best alpha: 0.5 -> 91.06%


In [32]:
# ── Alpha sweep chart ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))

colors = ['#E85D24' if a == alpha_df.iloc[0]['alpha'] else '#3B8BD4'
          for a in alpha_df['alpha']]
bars = ax.bar([str(a) for a in alpha_df['alpha']],
              alpha_df['best_accuracy'], color=colors, width=0.5)

for bar, val in zip(bars, alpha_df['best_accuracy']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{val:.2f}%', ha='center', fontsize=9)

ax.axhline(fp32_accuracy, color='gray', linestyle='--', linewidth=1.2,
           label=f'FP32 teacher ({fp32_accuracy:.2f}%)')
ax.set_xlabel('alpha (soft-loss weight)')
ax.set_ylabel('Best eval accuracy (%)')
ax.set_title(f'Effect of alpha on distillation recovery (T={best_T})')
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, 'alpha_sweep.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: results/alpha_sweep.png')

Saved: results/alpha_sweep.png


## §8 · Final Benchmark — all model variants

We now register every model variant with `ModelBenchmark` and produce
the definitive results table. Because some models are MNLI-adapted
(3-class) and some are SST-2 (2-class), we evaluate each on its
own task and note this in the table.

In [33]:
from src.benchmark import ModelBenchmark

bench = ModelBenchmark(device=CPU, warmup_runs=10, timed_runs=100)

# ── Helper: get size in MB ────────────────────────────────────────────────────
def model_size_mb(model):
    with tempfile.NamedTemporaryFile(suffix='.pt', delete=False) as f:
        torch.save(model.state_dict(), f.name)
        return os.path.getsize(f.name) / 1e6

# ── Register all variants (CPU for quantized models) ──────────────────────────

# 1. FP32 baseline (SST-2)
bench.register('FP32 baseline',      model_fp32.cpu().eval(), fp32_accuracy)

# 2. PTQ INT8 (SST-2)
bench.register('PTQ INT8',           model_ptq,               ptq_accuracy)

# 3. LoRA + MNLI (FP32 — LoRA adapters, MNLI task)
bench.register('LoRA FP32 (MNLI)',   model_merged.cpu().eval(), lora_mnli_accuracy)

# 4. LoRA + QAT INT8 (MNLI task, INT8)
bench.register('LoRA + QAT INT8',    model_qat,               qat_accuracy)

# 5. Distillation best (SST-2, FP32 LoRA student, best T)
bench.register('Distillation (best T)', trainer.student.cpu().eval(), distil_accuracy)

# ── Sample input (SST-2 compatible — shape works for all models) ──────────────
_sample = {
    'input_ids'      : torch.ones(1, MAX_LENGTH, dtype=torch.long),
    'attention_mask' : torch.ones(1, MAX_LENGTH, dtype=torch.long),
}

results_df = bench.run(_sample)
bench.print_table()
bench.save_csv(os.path.join(RESULTS_DIR, 'benchmark_table_nb02.csv'))


Running benchmarks...
  Warmup: 10 runs | Timed: 100 runs
  Device: cpu

  Benchmarking: FP32 baseline
    Size:          267.86 MB
    Accuracy:      91.06%
    Latency @bs=1: 69.62 ± 2.38 ms
    Latency @bs=32:1922.01 ± 94.12 ms
    Throughput:    14.4 / 16.6 samples/s

  Benchmarking: PTQ INT8
    Size:          138.72 MB
    Accuracy:      89.68%
    Latency @bs=1: 53.68 ± 3.85 ms
    Latency @bs=32:1698.04 ± 34.00 ms
    Throughput:    18.6 / 18.8 samples/s
    Speedup vs FP32: 1.30x

  Benchmarking: LoRA FP32 (MNLI)
    Size:          270.23 MB
    Accuracy:      66.02%
    Latency @bs=1: 71.43 ± 4.43 ms
    Latency @bs=32:1874.71 ± 29.87 ms
    Throughput:    14.0 / 17.1 samples/s
    Speedup vs FP32: 0.97x

  Benchmarking: LoRA + QAT INT8
    Size:          270.23 MB
    Accuracy:      64.40%
    Latency @bs=1: 70.59 ± 3.50 ms
    Latency @bs=32:1880.88 ± 31.68 ms
    Throughput:    14.2 / 17.0 samples/s
    Speedup vs FP32: 0.99x

  Benchmarking: Distillation (best T)
    Siz

In [34]:
# ── Comparison charts ─────────────────────────────────────────────────────────
fig = bench.plot_comparison(
    save_path = os.path.join(RESULTS_DIR, 'benchmark_comparison_nb02.png'),
    figsize   = (14, 8),
)
plt.show()
print('Saved: results/benchmark_comparison_nb02.png')

Comparison chart saved to: /kaggle/working/results/benchmark_comparison_nb02.png
Saved: results/benchmark_comparison_nb02.png


## §9 · Consolidated Results & Decision Narrative

This section produces the final table that spans **both notebooks** —
the complete picture from raw FP32 to the best compressed+fine-tuned model.

In [35]:
# ── Master results table (both notebooks combined) ───────────────────────────
print('\n' + '='*90)
print('MASTER RESULTS TABLE — DistilBERT compression & fine-tuning pipeline')
print('='*90)
print(f'{"Model":<30} {"Task":<8} {"Accuracy":>10} {"Size MB":>9} '
      f'{"Acc delta":>10} {"Compression":>12}')
print('-'*90)

rows = [
    # (label, task, accuracy, size_mb)
    ('FP32 baseline',          'SST-2', fp32_accuracy,      FP32_SIZE_MB),
    ('PTQ INT8',               'SST-2', ptq_accuracy,       ptq_size_mb),
    ('LoRA FP32 (r=8)',        'MNLI',  lora_mnli_accuracy, model_size_mb(model_merged.cpu())),
    ('LoRA + QAT INT8',        'MNLI',  qat_accuracy,       qat_size_mb),
    ('Distillation (best T)',  'SST-2', distil_accuracy,    model_size_mb(trainer.student.cpu())),
]

for label, task, acc, size in rows:
    delta = acc - fp32_accuracy
    ratio = f'{FP32_SIZE_MB/size:.1f}x' if size > 0 else '--'
    print(f'  {label:<28} {task:<8} {acc:>9.2f}%  {size:>8.2f}  '
          f'{delta:>+9.2f}%  {ratio:>11}')

print('='*90)

# Sweep results summary
print(f'\nTemperature sweep winner : T={best_T}  ->  {best_T_acc:.2f}%')
print(f'Alpha sweep winner       : alpha={best_alpha}  ->  {alpha_df.iloc[0]["best_accuracy"]:.2f}%')


MASTER RESULTS TABLE — DistilBERT compression & fine-tuning pipeline
Model                          Task       Accuracy   Size MB  Acc delta  Compression
------------------------------------------------------------------------------------------
  FP32 baseline                SST-2        91.06%    267.86      +0.00%         1.0x
  PTQ INT8                     SST-2        89.68%    138.72      -1.38%         1.9x
  LoRA FP32 (r=8)              MNLI         66.02%    270.23     -25.03%         1.0x
  LoRA + QAT INT8              MNLI         64.40%    139.31     -26.65%         1.9x
  Distillation (best T)        SST-2        90.94%    270.85      -0.11%         1.0x

Temperature sweep winner : T=4.0  ->  91.17%
Alpha sweep winner       : alpha=0.5  ->  91.06%


In [36]:
# ── List all saved artefacts ──────────────────────────────────────────────────
print('\nSaved artefacts (results/):')
for fname in sorted(os.listdir(RESULTS_DIR)):
    fpath = os.path.join(RESULTS_DIR, fname)
    kb = os.path.getsize(fpath) / 1e3
    print(f'  {fname:<50} {kb:>7.0f} KB')


Saved artefacts (results/):
  activation_histograms.png                               95 KB
  alpha_sweep.png                                         46 KB
  benchmark_comparison.png                               138 KB
  benchmark_comparison_nb02.png                          147 KB
  benchmark_table.csv                                      0 KB
  benchmark_table_nb02.csv                                 0 KB
  distilbert_fp32.onnx                                   775 KB
  distilbert_fp32.onnx.data                           267827 KB
  distilbert_int8.onnx                                   775 KB
  distilbert_int8.onnx.data                           267827 KB
  distillation_history.png                                81 KB
  gradient_statistics.png                                 99 KB
  lora_training_curves.png                                90 KB
  sensitivity_chart.png                                  223 KB
  temperature_sweep.png                                   42 KB
